# 🚀 進階版：用 AI 一起想出來的改進解法

**比賽**：[Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)

這份 notebook 是**課堂示範的第二步**：在 baseline（0.143）之上，做出一個更好的版本。重點不只是程式，而是**「怎麼用 AI 找到這些改進」**——下面每個改進，都是我們同時問了三個不同的 AI、再綜合出來的。

## 三個 AI 給的點子，怎麼收斂
我們把同一個問題（「baseline 0.143 卡在哪、怎麼往上推」）**分別**丟給三個 AI：

| 來源 | 角色 | 重點建議 |
|---|---|---|
| **木瓜溪**（OpenCode）| 快速腦力激盪 | U-Net 分割、**顯式偵測細胞分裂**、Kalman、軌跡篩選、真假細胞分類器 |
| **秀姑巒**（Gemini）| 廣搜＋多方案比較 | StarDist/Cellpose 分割、btrack/ultrack 追蹤、**分裂偵測**、分階段路線圖 |
| **立霧**（Codex）| web 研究＋論文來源 | watershed 拆黏連、global gap-closing、**division 規則**、metric-aware 後處理＋本地驗證 |

**三個 AI 各自獨立，卻一致指向同樣三件事** → 這種「不約而同」就是高信心的訊號。我們挑出其中**不需 GPU、不需網路、今天就能跑**的三個來實作：

1. **偵測升級**：用 watershed 把黏在一起的細胞拆開（baseline 的連通元件會把它們算成一顆）。
2. **細胞分裂偵測**：baseline 完全沒做，這 0.1× 的分數整碗端走；用一條簡單規則把它撈回來。
3. **斷軌補回（gap closing）**：細胞某一幀沒偵測到就斷掉，跨 2 幀把它接回來。

> 更強但更重的做法（Cellpose/StarDist 深度學習分割、btrack/ultrack 全域圖追蹤）寫在最後的「下一步路線圖」，那需要離線打包套件與 GPU，留給想衝高分的人。

> 🙏 改編自官方 Apache-2.0 baseline；改進方向綜合自團隊三個 AI 夥伴的建議（見文末來源）。

In [ ]:
import json, os
from collections import defaultdict

import blosc2
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter, distance_transform_edt
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

print('套件載入完成')

In [ ]:
# === 參數（baseline 的＋三個新加的）===
TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'
SCALE = np.array([1.625, 0.40625, 0.40625])  # (Z,Y,X) µm/voxel
DOWNSAMPLE = 4
PERCENTILE = 90
MAX_LINK_DISTANCE = 15.0   # µm：相鄰幀配對上限

# —— 新增 ——
WS_MIN_DISTANCE = 2        # watershed 種子點的最小間距（降採樣空間，voxel）
DIV_DISTANCE = 8.0         # µm：分裂的兩個子細胞要離母細胞夠近
GAP_DISTANCE = 20.0        # µm：跨 2 幀補軌的距離上限

def scaled_pairwise(A, B):
    """A:(P,3) B:(C,3) 體素座標 → (P,C) 微米距離矩陣。"""
    diff = A[:, None, :] - B[None, :, :]
    return np.sqrt(((diff * SCALE) ** 2).sum(axis=2))

print('參數設定完成')

## 改進 1：偵測升級 — watershed 拆開黏在一起的細胞

baseline 用「連通元件」找細胞，兩顆貼著的細胞會被算成一顆 → 少了 node、也少了該有的 edge。做法：二值化後算**距離轉換**，在距離圖的局部高點放**種子**，再用 **watershed** 把一團拆成多顆。

> 出處：立霧引 CTC 研究（marker-controlled watershed 對密集細胞有效）、秀姑巒的分割方案表。

In [ ]:
def read_zarr_meta(zarr_path):
    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        meta = json.load(f)
    return tuple(meta['shape']), np.dtype(meta['data_type'])


def read_timepoint(zarr_path, t, shape, dtype):
    chunk_path = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
    if not os.path.exists(chunk_path):
        return np.zeros(shape[1:], dtype=dtype)
    with open(chunk_path, 'rb') as f:
        decompressed = blosc2.decompress(f.read())
    return np.frombuffer(decompressed, dtype=dtype).reshape(shape[1:])


def detect_cells(vol):
    """watershed 版偵測：回傳每顆細胞質心 (z,y,x) 體素座標。"""
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    smoothed = uniform_filter(ds.astype(np.float32), size=3)
    binary = smoothed > np.percentile(smoothed, PERCENTILE)
    if not binary.any():
        return []

    dist = distance_transform_edt(binary)               # 離邊界越遠值越大
    peaks = peak_local_max(dist, min_distance=WS_MIN_DISTANCE, labels=binary)
    if len(peaks) == 0:
        return []
    markers = np.zeros(dist.shape, dtype=np.int32)       # 每個種子一個編號
    markers[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
    labels = watershed(-dist, markers, mask=binary)      # 從種子往外淹，淹到分界線

    centroids = []
    for i in range(1, int(labels.max()) + 1):
        coords = np.argwhere(labels == i)
        if len(coords):
            centroids.append(coords.mean(axis=0) * DOWNSAMPLE)  # 乘回原解析度
    return centroids

print('偵測函式（watershed 版）定義完成')

## 改進 2＋3：連線時加「細胞分裂」與「斷軌補回」

一個 dataset 的完整流程：
1. 每幀偵測 → 收 node。
2. **相鄰幀匈牙利配對**（baseline 的一對一連線）。
3. **分裂偵測**：配對完，若某個沒被配到的子細胞，緊鄰一個「已經有一個小孩」的母細胞 → 把它當第二個小孩接上去（母細胞就有 2 條 outgoing edge ＝ 一次分裂）。
4. **斷軌補回**：把「軌跡尾端」（沒有 outgoing 的點）跨 2 幀接到「軌跡開頭」（沒有 incoming 的點）。

> 出處：分裂規則＝立霧#7（引 Linajea）＋木瓜溪#2；gap closing＝立霧#5（引 btrack/TrackMate）。

In [ ]:
def track_one_dataset(folder_name, zarr_path):
    """回傳這個 dataset 的 (node_rows, edge_rows, n_divisions)。"""
    shape, dtype = read_zarr_meta(zarr_path)
    n_t = shape[0]

    node_rows, edge_rows = [], []
    frame_ids, frame_xyz = [], []        # 每幀的 node_id 與座標
    node_id = 1

    # ---- 1) 逐幀偵測，收 node ----
    for t in range(n_t):
        vol = read_timepoint(zarr_path, t, shape, dtype)
        cents = detect_cells(vol)
        ids, xyz = [], []
        for c in cents:
            z, y, x = int(round(c[0])), int(round(c[1])), int(round(c[2]))
            node_rows.append({'dataset': folder_name, 'row_type': 'node', 'node_id': node_id,
                              't': t, 'z': z, 'y': y, 'x': x, 'source_id': -1, 'target_id': -1})
            ids.append(node_id); xyz.append(c); node_id += 1
        frame_ids.append(ids)
        frame_xyz.append(np.array(xyz) if xyz else np.empty((0, 3)))

    has_in, out_count = set(), defaultdict(int)

    def add_edge(s, d):
        edge_rows.append({'dataset': folder_name, 'row_type': 'edge', 'node_id': -1,
                          't': -1, 'z': -1, 'y': -1, 'x': -1, 'source_id': s, 'target_id': d})

    # ---- 2) 相鄰幀配對 + 3) 分裂 ----
    for t in range(n_t - 1):
        pid, pc = frame_ids[t], frame_xyz[t]
        cid, cc = frame_ids[t + 1], frame_xyz[t + 1]
        if len(pid) == 0 or len(cid) == 0:
            continue
        D = scaled_pairwise(pc, cc)
        rows, cols = linear_sum_assignment(D)
        matched_parent, matched_child = set(), set()
        for ri, ci in zip(rows, cols):
            if D[ri, ci] <= MAX_LINK_DISTANCE:
                add_edge(pid[ri], cid[ci]); matched_parent.add(ri); matched_child.add(ci)
                has_in.add(cid[ci]); out_count[pid[ri]] += 1
        # 分裂：沒配到的子細胞 → 最近的「已配對」母細胞，且夠近、母細胞還沒滿 2 個小孩
        for ci in range(len(cid)):
            if ci in matched_child:
                continue
            d = np.sqrt((((pc - cc[ci]) * SCALE) ** 2).sum(axis=1))
            j = int(np.argmin(d))
            if j in matched_parent and d[j] <= DIV_DISTANCE and out_count[pid[j]] < 2:
                add_edge(pid[j], cid[ci]); has_in.add(cid[ci]); out_count[pid[j]] += 1

    has_out = set(out_count.keys())

    # ---- 4) 斷軌補回（t 的尾端 → t+2 的開頭）----
    for t in range(n_t - 2):
        ends = [(i, nid) for i, nid in enumerate(frame_ids[t]) if nid not in has_out]
        starts = [(j, nid) for j, nid in enumerate(frame_ids[t + 2]) if nid not in has_in]
        if not ends or not starts:
            continue
        ec = frame_xyz[t][[i for i, _ in ends]]
        sc = frame_xyz[t + 2][[j for j, _ in starts]]
        D = scaled_pairwise(ec, sc)
        rows, cols = linear_sum_assignment(D)
        for ri, ci in zip(rows, cols):
            if D[ri, ci] <= GAP_DISTANCE:
                s_nid, t_nid = ends[ri][1], starts[ci][1]
                add_edge(s_nid, t_nid); has_out.add(s_nid); has_in.add(t_nid)

    n_div = sum(1 for v in out_count.values() if v >= 2)
    return node_rows, edge_rows, n_div

print('追蹤函式（含分裂＋補軌）定義完成')

## 跑完整測試集，產生 `submission.csv`

In [ ]:
test_folder_names = sorted(
    d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr')
)

all_rows = []
for folder_name in test_folder_names:
    zarr_path = os.path.join(TEST_DIR, folder_name + '.zarr')
    node_rows, edge_rows, n_div = track_one_dataset(folder_name, zarr_path)
    all_rows.extend(node_rows)
    all_rows.extend(edge_rows)
    print(f'{folder_name}: {len(node_rows)} 顆細胞, {len(edge_rows)} 條連線, {n_div} 次分裂')

print('\n總列數：', len(all_rows))

In [ ]:
submission = pd.DataFrame(all_rows)
submission.insert(0, 'id', range(len(submission)))
submission = submission[['id', 'dataset', 'row_type', 'node_id',
                         't', 'z', 'y', 'x', 'source_id', 'target_id']]
submission.to_csv('submission.csv', index=False)

n_nodes = (submission.row_type == 'node').sum()
n_edges = (submission.row_type == 'edge').sum()
print(f'已寫出 submission.csv：{n_nodes} node 列、{n_edges} edge 列')
print('對照官方 baseline：完全沒有分裂、也沒有補軌——我們這版兩個都有了。')
submission.head()

## 下一步路線圖（想再往 0.687 衝的人）

三個 AI 一致認為，要再大幅提升就得換引擎——這層需要**離線打包套件 + GPU**，難度較高：

| 階段 | 換什麼 | 工具／論文 |
|---|---|---|
| 分割升級 | 深度學習實例分割取代閾值 | [Cellpose](https://cellpose.readthedocs.io/en/latest/do3d.html)、[StarDist](https://github.com/stardist/stardist) |
| 追蹤升級 | 全域圖／多假設追蹤取代相鄰配對 | [btrack](https://btrack.readthedocs.io/)、[ultrack](https://arxiv.org/abs/2308.04526)（CZ Biohub 自家）、[Trackastra](https://github.com/weigertlab/trackastra) |
| 分裂升級 | 學習式分裂偵測 | [Linajea](https://arxiv.org/abs/2208.11467) |
| 調參 | 用訓練集的 `.geff` 做**本地驗證**，掃門檻/距離 | metric-aware 後處理（立霧 #8） |

> ⚠️ Kaggle 提交關閉網路：套件要先 `pip download` 成 wheel 做成 Dataset 掛進來、預訓練權重也要先放進快取（見秀姑巒的部署筆記）。

**最實在的下一步**：先用訓練集做一個本地驗證分數，這樣每改一版不用浪費提交次數就知道有沒有變好——這是把分數穩定往上推的關鍵習慣。